# Exp-1 BNN

This notebook uses `src/` modules for local execution.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from sklearn.metrics import mean_squared_error
from pymoo.operators.sampling.lhs import LHS

from src.opt_problem import build_problem
from src.data import generate_data
from src.metrics import get_metrics
from src.survival import Survival_standard
from src.experiment import run_experiment
from src.other_functions import mean_std


In [ ]:
from src.models import BNN_Ensemble, surrogate_pred_mean_std

problem_name = "dtlz1"
n_var = 10
n_obj = 2
n_gen = 50
pop_size = 100

problem = build_problem(problem_name, n_var=n_var, n_obj=n_obj)

X_train, y_train, X_val, y_val, X_test, y_test = generate_data(
    problem,
    sample_size=500,
    sampling=LHS(),
    train_seed=42,
    val_size=200,
    test_size=1000,
    test_seed=1,
)

model_f1 = BNN_Ensemble()
model_f2 = BNN_Ensemble()
model_f1.fit(X_train, y_train[:, 0])
model_f2.fit(X_train, y_train[:, 1])

pred_mean, pred_std, *_ = surrogate_pred_mean_std(model_f1, model_f2, X_test)
print(f"MSE: {mean_squared_error(y_test, pred_mean):.2e}")


In [ ]:
hv, igd_plus, obj_min, obj_max, ref = get_metrics(problem_name, problem, n_var=n_var, n_obj=n_obj)

results = run_experiment(
    problem=problem,
    problem_name=problem_name,
    n_gen=n_gen,
    pop_size=pop_size,
    use_surrogate="BNN_uncertainty",
    model_f1=model_f1,
    model_f2=model_f2,
    survival_function=Survival_standard(),
    obj_min=obj_min,
    obj_max=obj_max,
    hv=hv,
    igd_plus=igd_plus,
    use_callback=False,
    seeds=range(1, 3),
)

print("MSE mean/std:", mean_std(results["mse_list"]))
print("IGD+ mean/std:", mean_std(results["igd_list"]))
print("HV(real) mean/std:", mean_std(results["hv_real_list"]))
